# Random Forest Regression

In [ ]:
# importing libs:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.calibration import LabelEncoder
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from category_encoders.target_encoder import TargetEncoder
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
df = pd.read_csv("../../../../../data/Used_Car_Price_Prediction.csv")
df.head()

## EDA

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns
non_numeric_cols = df.select_dtypes(exclude='number').columns
numeric_cols

In [ ]:
non_numeric_cols

In [ ]:
for col in non_numeric_cols:
   print(df[col].value_counts(),"\n")

In [ ]:
cols = df.columns
for col in cols:
   print(col ,':',df[col].count(),":" ,len(df[col].unique()),"\n")
   print(df[col].value_counts(),"\n")

In [ ]:
df.head(3)

In [ ]:
df.isnull().sum()

In [ ]:
df.drop(columns=['original_price'], inplace = True) ## to many NaN

In [ ]:
df.describe().T

In [ ]:
num_col = df.select_dtypes(include = 'number')
plt.figure(figsize=(12,40))
index = 1
for col in num_col:
    plt.subplot(11,4,index)
    sns.kdeplot(df[col])
    index +=1

In [ ]:
plt.figure(figsize=(8,6))
sns.lineplot(df['sale_price'])
plt.title('Lineplot for Sale Price')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(num_col.corr(),annot=True,cmap='Blues')

In [ ]:
sns.boxplot(df)

In [ ]:
def cap_outliers(df):
    num = df.select_dtypes(include = 'number')
    for col in num:
        Q1, Q3 = df[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        df[col] = df[col].clip(lower, upper)
    return df 

df = cap_outliers(df)

In [ ]:
sns.pairplot(num_col)

In [ ]:
df.drop(columns='ad_created_on')

## Data Splitting

In [ ]:
X = df.drop(columns = 'sale_price')
y = df['sale_price']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 42
)

In [ ]:
df.isnull().sum()[df.isnull().sum() > 1]

In [ ]:
df.head()

In [ ]:
# Get columns that have null values
null_cols = df.columns[df.isnull().any()]

# Loop through those columns and print the desired information
for col in null_cols:
    print(col, ':', df[col].count(), ":", len(df[col].unique()), "\n")
    print(df[col].value_counts(), "\n")
# body_type, transmission, registered_state, source,car_availability,car_rating, 

In [ ]:
df[num_col.columns]

In [ ]:
# print(num_col)
for col in num_col:
    if col in null_cols:
        print(col, ':', df[col].count(), ":", len(df[col].unique()), "\n")
        print(df[col].value_counts(), "\n")


In [ ]:
from sklearn.calibration import LabelEncoder


numerical_col = ['yr_mfr',
 'kms_run',
 'times_viewed',
 'total_owners',
 'broker_quote',
 'emi_starts_from',
 'booking_down_pymnt']

# since we have all continuous num values:
numeric = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_features = df.select_dtypes(include=['object', 'category']).columns.tolist()

# Categorical pipeline
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # fill missing with mode
    ('label', LabelEncoder())
])


preprocessor = ColumnTransformer([
    ('numeric', numeric, numerical_col),
    ('cat', cat_pipeline, cat_features)
])

In [ ]:
df[cat_features]

In [ ]:
models = {
    "Random Forest": RandomForestRegressor(),
}

In [ ]:
preprocessor

In [ ]:
pipelines = {name: Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
]) for name, model in models.items()}


In [ ]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor())
])

In [ ]:
pipelines

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.25, random_state=67
    )

In [ ]:
X_train

In [ ]:
for name, cols in [('num', numerical_col),
                   ('categorical', cat_features)]:
    print(f"{name} columns:", cols)

In [ ]:
X_train_transformed_dense = X_train_transformed.toarray()
pd.DataFrame(X_train_transformed_dense)


In [ ]:
X_train_transformed = preprocessor.fit_transform(X_train)
# X_train_transformed=pd.DataFrame(X_train)
pd.DataFrame(X_train_transformed)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_pred, y_test)
mse = mean_squared_error(y_pred, y_test)
r2 = r2_score(y_pred, y_test)

In [ ]:
print("Mean Absolute Error : ", mae)
print("Mean Squared Error : ", mse)
print("R2 Score : ", 100 * r2)

In [ ]:
scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')  # R² score

print("Cross-validation scores (R²):", scores)
print("Mean R²:", 100 * np.mean(scores))
print("Standard Deviation:", np.std(scores))

In [ ]:
import pandas as pd

# Transform the training data using the preprocessor
X_train_transformed = preprocessor.transform(X_train)

# Convert the transformed data to a dense matrix (if it's sparse)
X_train_transformed_dense = X_train_transformed.toarray() if hasattr(X_train_transformed, 'toarray') else X_train_transformed

# Get the transformed column names (numerical columns + one-hot encoded categorical columns)
# Numerical columns
numerical_columns = numerical_col
# OneHotEncoded categorical columns
categorical_columns = list(preprocessor.transformers_[1][1].named_steps['onehot'].get_feature_names_out(cat_cols))

# Combine column names
transformed_columns = numerical_columns + categorical_columns

# Convert to DataFrame with correct column names
X_train_transformed_df = pd.DataFrame(X_train_transformed_dense, columns=transformed_columns)

# View the transformed training data
print("Transformed Training Data:")
X_train_transformed_df.head()


In [ ]:
print(X_train_transformed_df.columns)

In [ ]:
# Define the models
models = {
    "Random Forest": RandomForestRegressor(),
    "Linear Regression": LinearRegression(),
    "SVR": SVR(),
    "Decision Tree": DecisionTreeRegressor()
}

# Create the pipelines for each model
pipelines = {name: Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
]) for name, model in models.items()}

In [ ]:
X = df.drop(columns='sale_price')
y = df['sale_price']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=67)

# Store results for each model
results = {}

In [ ]:
for name, pipe in pipelines.items():
    print(f"Training {name}...")

    # Train the model
    pipe.fit(X_train, y_train)

    # Predict on the test set
    y_pred = pipe.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Cross-validation scores
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='r2')

    # Store the results
    results[name] = {
        'MAE': mae,
        'MSE': mse,
        'R²': r2,
        'CV R² Mean': np.mean(cv_scores),
        'CV R² Std': np.std(cv_scores)
    }

    # Print results for each model
    print(f"--- {name} ---")
    print(f"Mean Absolute Error : {mae}")
    print(f"Mean Squared Error : {mse}")
    print(f"R² Score : {r2 * 100}")  # R² as percentage
    print(f"Cross-validation R² Mean: {np.mean(cv_scores) * 100}")  # Cross-validation R² as percentage
    print(f"Cross-validation R² Std: {np.std(cv_scores)}")
    print("-" * 40)

In [ ]:
# Display all results
print("All Model Results:")
for model_name, metrics in results.items():
    print(f"{model_name}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")
    print("-" * 40)

In [ ]:
import pandas as pd

# Transform the training data using the preprocessor
X_train_transformed = preprocessor.transform(X_train)

# Convert the transformed data to a dense matrix (if it's sparse)
X_train_transformed_dense = X_train_transformed.toarray() if hasattr(X_train_transformed, 'toarray') else X_train_transformed

# Get the transformed column names (numerical columns + one-hot encoded categorical columns)
# Numerical columns
numerical_columns = numerical_col
# OneHotEncoded categorical columns
categorical_columns = list(preprocessor.transformers_[1][1].named_steps['onehot'].get_feature_names_out(cat_cols))

# Combine column names
transformed_columns = numerical_columns + categorical_columns

# Convert to DataFrame with correct column names
X_train_transformed_df = pd.DataFrame(X_train_transformed_dense, columns=transformed_columns)

# View the transformed training data
print("Transformed Training Data:")
X_train_transformed_df.head()


In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Define the columns for continuous and categorical data
numerical_col = ['yr_mfr', 'kms_run', 'times_viewed', 'total_owners', 'broker_quote', 'emi_starts_from', 'booking_down_pymnt']
cat_cols = ['car_name', 'fuel_type', 'city', 'body_type', 'transmission', 'variant', 'registered_city', 'registered_state', 
            'rto', 'source', 'make', 'model', 'car_availability', 'car_rating', 'ad_created_on', 'fitness_certificate']

# Preprocessing pipeline
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # Handle missing data
    ('scaler', StandardScaler())  # Normalize the continuous features
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Handle missing categorical data
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))  # One-hot encoding
])

preprocessor = ColumnTransformer([
    ('cont', numeric_pipeline, numerical_col),
    ('cat', cat_pipeline, cat_cols)
])

# Define the models
models = {
    "Random Forest": RandomForestRegressor(),
    "Linear Regression": LinearRegression(),
    "SVR": SVR(),
    "Decision Tree": DecisionTreeRegressor()
}

# Create the pipelines for each model
pipelines = {name: Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
]) for name, model in models.items()}

# Split data into train/test sets
X = df.drop(columns='sale_price')
y = df['sale_price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=67)

# Store results for each model
results = {}

# Loop over each model pipeline
for name, pipe in pipelines.items():
    print(f"Training {name}...")

    # Train the model
    pipe.fit(X_train, y_train)

    # Predict on the test set
    y_pred = pipe.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Cross-validation scores
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='r2')

    # Store the results
    results[name] = {
        'MAE': mae,
        'MSE': mse,
        'R²': r2,
        'CV R² Mean': np.mean(cv_scores),
        'CV R² Std': np.std(cv_scores)
    }

    # Print results for each model
    print(f"--- {name} ---")
    print(f"Mean Absolute Error : {mae}")
    print(f"Mean Squared Error : {mse}")
    print(f"R² Score : {r2 * 100}")  # R² as percentage
    print(f"Cross-validation R² Mean: {np.mean(cv_scores) * 100}")  # Cross-validation R² as percentage
    print(f"Cross-validation R² Std: {np.std(cv_scores)}")
    print("-" * 40)

# Display all results
print("All Model Results:")
for model_name, metrics in results.items():
    print(f"{model_name}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")
    print("-" * 40)


In [ ]:
X_train_transformed = preprocessor.fit_transform(X_train)

# Create DataFrame with transformed data
X_train_transformed = pd.DataFrame(X_train_transformed)

In [ ]:
X_train

In [ ]:
X_train_transformed

In [ ]:
X_test_transformed = pd.DataFrame(preprocessor.transform(X_test))

In [ ]:
X_test_transformed

## Training

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

def get_classification_metrics(y_true, y_pred, y_proba=None, model_name=None, verbose=True):
    """
    Calculate standard classification metrics and optionally print them.

    Parameters:
    -----------
    y_true : array-like
        True labels
    y_pred : array-like
        Predicted labels
    y_proba : array-like, optional
        Predicted probabilities for positive class (for ROC AUC)
    model_name : str, optional
        Name of the model for printing
    verbose : bool
        Whether to print the metrics

    Returns:
    --------
    metrics : dict
        Dictionary containing accuracy, precision, recall, f1, roc_auc, confusion_matrix
    """
    metrics = {}
    metrics['accuracy'] = accuracy_score(y_true, y_pred)
    metrics['precision'] = precision_score(y_true, y_pred)
    metrics['recall'] = recall_score(y_true, y_pred)
    metrics['f1'] = f1_score(y_true, y_pred)
    if y_proba is not None:
        metrics['roc_auc'] = roc_auc_score(y_true, y_proba)
    else:
        metrics['roc_auc'] = None

    metrics['confusion_matrix'] = confusion_matrix(y_true, y_pred)
    if verbose:
        if model_name:
            print(f"--- {model_name} ---")
        print(f"Accuracy : {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall   : {metrics['recall']:.4f}")
        print(f"F1-score : {metrics['f1']:.4f}")
        print(f"ROC AUC  : {metrics['roc_auc']}")
        print("Confusion Matrix:")
        print(metrics['confusion_matrix'])
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred))
        print("-"*40)
    return metrics


In [ ]:
results = {}

for name, pipe in pipelines.items():
    print(f"Training {name}...")

    # Train the model
    pipe.fit(X_train, y_train)

    # TRAIN metrics
    y_train_pred = pipe.predict(X_train)
    try:
        y_train_proba = pipe.predict_proba(X_train)[:, 1]
    except AttributeError:
        y_train_proba = None
    train_metrics = get_classification_metrics(y_train, y_train_pred, y_train_proba,verbose=True)

    # TEST metrics
    y_test_pred = pipe.predict(X_test)
    try:
        y_test_proba = pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None
    test_metrics = get_classification_metrics(y_test, y_test_pred, y_test_proba,verbose=True)

    results[name] = {'train': train_metrics, 'test': test_metrics}


In [ ]:
cat_cols = ['car_name',
 'fuel_type',
 'city',
 'body_type',
 'transmission',
 'variant',
 'registered_city',
 'registered_state',
 'rto',
 'source',
 'make',
 'model',
 'car_availability',
 'car_rating',
 'ad_created_on',
 'fitness_certificate']

onehot = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown = 'ignore'))
])

## Redo!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

In [ ]:
df.head()

In [ ]:
df['model'].unique()

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df['fitness_certificate'] = df['fitness_certificate'].astype('boolean')

In [ ]:
df_bool = df.select_dtypes('boolean')
df_bool

In [ ]:
df['car_availability'].unique()

In [ ]:
df.drop(columns=['original_price'], inplace = True) ## to many NaN

In [ ]:
df_obj = df.select_dtypes(include='object')
df_obj_one = []
for col in df_obj.columns:
    unique_vals = df[col].dropna().unique()
    n_unique = len(unique_vals)

    if n_unique < 10:
        print(f"\nColumn: {col}")
        df_obj_one.append(col)
        print(f"Number of unique values: {n_unique}")
        print("Unique values:")
        print(unique_vals)
print(df_obj_one)

In [ ]:
x = 5

df_obj = df.select_dtypes(include='object')

for col in df_obj.columns:
    n_unique = df[col].nunique(dropna=True)
    
    if n_unique < x:
        print(f"\nColumn: {col}")
        print(f"Number of unique values: {n_unique}")
        print(df[col].value_counts())


In [ ]:
df_obj_one # to be one hot encoded
df_obj_target = [x for x in df_obj.columns if x not in df_obj_one]
df_obj_target

In [ ]:
df.describe().T

In [ ]:
num_col = df.select_dtypes(include = 'number')
plt.figure(figsize=(12,40))
index = 1
for col in num_col:
    plt.subplot(11,4,index)
    sns.kdeplot(df[col])
    index +=1

In [ ]:
plt.figure(figsize=(8,6))
sns.lineplot(df['sale_price'])
plt.title('Lineplot for Sale Price')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(num_col.corr(),annot=True,cmap='Blues')

In [ ]:
X = df.drop(columns = 'sale_price')
y = df['sale_price']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 42
)

In [ ]:
# now we make the pipelines again:

num_features = X.select_dtypes(include='number')
num_features = num_features.columns
num_features # numeric

In [ ]:
df_obj_target # categorical with >10 unique values

In [ ]:
df_obj_one # categorical with <10 unique values

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipeline_one = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one', OneHotEncoder(handle_unknown='ignore',drop='first'))
])
cat_pipeline_target = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target', TargetEncoder(handle_unknown='ignore',smoothing=5)),
    ('scaler', StandardScaler())
])

In [ ]:
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, num_features),
    ('cat_one',cat_pipeline_one,df_obj_one),
    ('cat_target',cat_pipeline_target,df_obj_target)
])

In [ ]:
models = {
    "Random Forest": RandomForestRegressor(),
}

In [ ]:
preprocessor

In [ ]:
pipelines = {name: Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
]) for name, model in models.items()}

In [ ]:
pipelines

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.25, random_state=67
    )

In [ ]:
X_train_transformed = preprocessor.fit_transform(X_train, y_train)
# X_train_transformed=pd.DataFrame(X_train)
mew = pd.DataFrame(X_train_transformed)
mew.head()

In [ ]:
def get_regression_metrics(y_true, y_pred, model_name=None, verbose=True, plot=True):
    """
    Calculate standard regression metrics and optionally print and visualize them.

    Parameters
    ----------
    y_true : array-like
        True target values
    y_pred : array-like
        Predicted target values
    model_name : str, optional
        Name of the model (for printing)
    verbose : bool
        Whether to print metrics
    plot : bool
        Whether to plot visualizations

    Returns
    -------
    metrics : dict
        Dictionary containing MAE, MSE, RMSE, R2
    """

    # Metrics
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    metrics = {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2
    }

    if verbose:
        if model_name:
            print(f"--- {model_name} ---")
        print(f"MAE  : {mae:.2f}")
        print(f"MSE  : {mse:.2f}")
        print(f"RMSE : {rmse:.2f}")
        print(f"R2   : {r2:.4f}")
        print("-"*40)

    # Visualizations
    if plot:
        plt.figure(figsize=(16,5))

        # 1️⃣ True vs Predicted Scatter
        plt.subplot(1,2,1)
        sns.scatterplot(x=y_true, y=y_pred, alpha=0.6)
        plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', linewidth=2)
        plt.xlabel("Actual Values")
        plt.ylabel("Predicted Values")
        plt.title(f"Actual vs Predicted {'(' + model_name + ')' if model_name else ''}")

        # 2️⃣ Residual Plot
        plt.subplot(1,2,2)
        residuals = y_true - y_pred
        sns.histplot(residuals, kde=True, bins=30, color='orange')
        plt.xlabel("Residuals")
        plt.title(f"Residuals Distribution {'(' + model_name + ')' if model_name else ''}")

        plt.tight_layout()
        plt.show()

    return metrics


In [ ]:
# Dictionary to store results
results = {}

for name, pipe in pipelines.items():
    print(f"Training {name}...\n")

    # Train the model
    pipe.fit(X_train, y_train)

    # -------------------------
    # TRAIN metrics
    # -------------------------
    y_train_pred = pipe.predict(X_train)
    train_metrics = get_regression_metrics(
        y_train,
        y_train_pred,
        model_name=f"{name} (Train)",
        verbose=True,
        plot=True
    )

    # -------------------------
    # TEST metrics
    # -------------------------
    y_test_pred = pipe.predict(X_test)
    test_metrics = get_regression_metrics(
        y_test,
        y_test_pred,
        model_name=f"{name} (Test)",
        verbose=True,
        plot=True
    )
    # Store metrics in dictionary
    results[name] = {
        'train': train_metrics,
        'test': test_metrics
    }


_________________________________________________________________________________
### **OverFitting** *so now we try the same data with multiple model then select the best model and hyper parameter tune them!!*
_________________________________________________________________________________

In [ ]:
X_train_transformed = preprocessor.fit_transform(X_train, y_train)
print(pd.DataFrame(X_train_transformed).isna().sum().sum())  # should be 0

X_test_transformed = preprocessor.transform(X_test)
print(pd.DataFrame(X_test_transformed).isna().sum().sum())  # should be 0


In [ ]:
# Columns in X_train
train_cols = set(X_train.columns)

# Columns in X_test
test_cols = set(X_test.columns)

# Columns missing in test
missing_in_test = train_cols - test_cols
print("Columns in train but missing in test:", missing_in_test)


In [ ]:
# Numeric
print("Missing numeric columns:", [c for c in num_features if c not in X_test.columns])

# One-hot categorical
print("Missing one-hot columns:", [c for c in df_obj_one if c not in X_test.columns])

# Target categorical
print("Missing target-encoded columns:", [c for c in df_obj_target if c not in X_test.columns])


In [ ]:
X_test_transformed = preprocessor.transform(X_test)
na_count = pd.DataFrame(X_test_transformed).isna().sum()
print("NaNs per column in transformed test set:")
print(na_count[na_count > 0])


In [ ]:
X_test_nan_counts = X_test.isna().sum()
print(X_test_nan_counts[X_test_nan_counts > 0])

In [ ]:
import category_encoders as ce

for col in df_obj_target:
    missing_after_te = X_test[col].isna().sum()
    print(f"{col}: {missing_after_te} NaNs in test set before encoding")


In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipeline_one = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one', OneHotEncoder(handle_unknown='ignore',drop='first'))
])
cat_pipeline_target = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('target', TargetEncoder(handle_unknown='ignore',smoothing=5)),
    ('final_imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, num_features),
    ('cat_one',cat_pipeline_one,df_obj_one),
    ('cat_target',cat_pipeline_target,df_obj_target)
])

In [ ]:
models = {
    "Random Forest": RandomForestRegressor(),
    "Linear Regression": LinearRegression(),
    "Ridge":Ridge(alpha=1.0),
    "Lasso":Lasso(),
    "ElasticNet":ElasticNet(),
    "SVR": SVR(),
    "Decision Tree": DecisionTreeRegressor()
}
pipelines = {name: Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
]) for name, model in models.items()}

In [ ]:
X_train_transformed = preprocessor.fit_transform(X_train, y_train)
X_test_transformed = preprocessor.transform(X_test)

print("Train NaNs:", pd.DataFrame(X_train_transformed).isna().sum().sum())
print("Test NaNs :", pd.DataFrame(X_test_transformed).isna().sum().sum())


In [ ]:
pd.DataFrame(X_train_transformed)

In [ ]:
pd.DataFrame(X_test_transformed)

In [ ]:
# Dictionary to store results
results = {}

for name, pipe in pipelines.items():
    print(f"Training {name}...\n")

    # Train the model
    pipe.fit(X_train, y_train)

    # -------------------------
    # TRAIN metrics
    # -------------------------
    y_train_pred = pipe.predict(X_train)
    train_metrics = get_regression_metrics(
        y_train,
        y_train_pred,
        model_name=f"{name} (Train)",
        verbose=True,
        plot=False
    )

    # -------------------------
    # TEST metrics
    # -------------------------
    y_test_pred = pipe.predict(X_test)
    test_metrics = get_regression_metrics(
        y_test,
        y_test_pred,
        model_name=f"{name} (Test)",
        verbose=True,
        plot=False
    )
    # Store metrics in dictionary
    results[name] = {
        'train': train_metrics,
        'test': test_metrics
    }


### Got decent Results in Ridge, Lasso, ElasticNet and Random Forest and Decision Tree so lets Hyperparameter Tune them

In [ ]:
models = {
    "Random Forest": RandomForestRegressor(),
    "Ridge":Ridge(),
    "Lasso":Lasso(),
    "ElasticNet":ElasticNet(),
    "Decision Tree": DecisionTreeRegressor()
}
pipelines = {name: Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
]) for name, model in models.items()}

In [ ]:
# Decision Tree Regressor
dt_params = {
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "criterion": ["squared_error", "friedman_mse", "absolute_error"]
}

# Random Forest Regressor
rf_params = {
    "n_estimators": [100, 200, 500, 1000],
    "max_depth": [None, 5, 8, 10, 15],
    "max_features": ["sqrt", "log2", 5, 7, 8],
    "min_samples_split": [2, 8, 15, 20],
    "min_samples_leaf": [1, 2, 4],
    "bootstrap": [True, False]
}

# Ridge Regressor
ridge_params = {
    "alpha": [0.01, 0.1, 1, 10, 100],
    "solver": ["auto", "svd", "cholesky", "saga"]
}

# Lasso Regressor
lasso_params = {
    "alpha": [0.01, 0.1, 1, 10, 100],
    "max_iter": [1000, 5000, 10000],
    "selection": ["cyclic", "random"]
}

# ElasticNet Regressor
elastic_params = {
    "alpha": [0.01, 0.1, 1, 10, 100],
    "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
    "max_iter": [1000, 5000, 10000],
    "selection": ["cyclic", "random"]
}


In [ ]:
randomcv_models = [
    ("Decision Tree", DecisionTreeRegressor(), dt_params),
    ("Random Forest", RandomForestRegressor(random_state=42), rf_params),
    ("Ridge", Ridge(), ridge_params),
    ("Lasso", Lasso(), lasso_params),
    ("ElasticNet", ElasticNet(), elastic_params)
]


In [ ]:
model_param = {}
for name, model, params in randomcv_models:
    pipe = Pipeline([
        ('preprocessor', preprocessor),  # your preprocessing pipeline
        ('model', model)
    ])
    # Prefix params for Pipeline
    params_prefixed = {f"model__{key}": value for key, value in params.items()}

    # RandomizedSearchCV for regression
    random = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=params_prefixed,
        n_iter=50,           # adjust for speed
        cv=3,
        verbose=2,
        n_jobs=-1,
        random_state=42,
        scoring='r2'          # regression metric
    )
    random.fit(X_train, y_train)
    model_param[name] = random.best_params_

# Print best params
for model_name in model_param:
    print(f'--------------Best Param for {model_name}------------------')
    print(model_param[model_name])

In [ ]:
# Dictionary to store pipelines with best params
pipelines_best = {}

for name, (_, model) in zip([m[0] for m in randomcv_models], [(m[1], m[1]) for m in randomcv_models]):
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    # Set the best params from RandomizedSearchCV
    pipe.set_params(**model_param[name])
    pipelines_best[name] = pipe

In [ ]:
# Dictionary to store results
results = {}

for name, pipe in pipelines_best.items():
    print(f"Training {name} with best params...\n")

    # -------------------------
    # TRAIN metrics
    # -------------------------
    pipe.fit(X_train, y_train)
    y_train_pred = pipe.predict(X_train)
    train_metrics = get_regression_metrics(
        y_train,
        y_train_pred,
        model_name=f"{name} (Train)",
        verbose=True,
        plot=False
    )

    # -------------------------
    # TEST metrics
    # -------------------------
    y_test_pred = pipe.predict(X_test)
    test_metrics = get_regression_metrics(
        y_test,
        y_test_pred,
        model_name=f"{name} (Test)",
        verbose=True,
        plot=False
    )

    # Store metrics
    results[name] = {
        'train': train_metrics,
        'test': test_metrics
    }

In [ ]:
#Dictionary to store results
results = {}

for name, pipe in pipelines_best.items():
    print(f"Training {name} with best params...\n")


    # -------------------------
    # TEST metrics
    # -------------------------
    y_test_pred = pipe.predict(X_test)
    test_metrics = get_regression_metrics(
        y_test,
        y_test_pred,
        model_name=f"{name} (Test)",
        verbose=True,
        plot=True
    )

    # Store metrics
    results[name] = {
        'train': train_metrics,
        'test': test_metrics
    }

Best performers: Lasso, ElasticNet, and Random Forest (after tuning).

Decision Tree: overfits slightly, test performance dropped after hyperparameter tuning.

Ridge: slightly improved after tuning.

Hyperparameter tuning success: Linear/regularized models benefited most, tree-based models benefit but require careful CV selection.

_______________________________________________________________________
**Unrealistic results! REDO**
_______________________________________________________________________

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

randomcv_models = [
    ("Decision Tree", DecisionTreeRegressor(random_state=42), dt_params),
    ("Random Forest", RandomForestRegressor(random_state=42), rf_params),
    ("Ridge", Ridge(), ridge_params),
    ("Lasso", Lasso(), lasso_params),
    ("ElasticNet", ElasticNet(), elastic_params)
]

# store fitted best estimators and best params
fitted_best_estimators = {}
best_params = {}

for name, model, params in randomcv_models:
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    params_prefixed = {f"model__{k}": v for k, v in params.items()}

    random_search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=params_prefixed,
        n_iter=50,
        cv=3,
        verbose=2,
        n_jobs=-1,
        random_state=42,
        scoring='r2',
        refit=True  # ensure best_estimator_ is fitted on the whole X_train
    )

    random_search.fit(X_train, y_train)       # fitted on X_train folds
    print(f"BEST for {name}: {random_search.best_params_}")
    best_params[name] = random_search.best_params_
    # best_estimator_ is already a Pipeline fitted on the entire X_train
    fitted_best_estimators[name] = random_search.best_estimator_

# Evaluate each fitted best estimator
results = {}
for name, fitted_pipe in fitted_best_estimators.items():
    print(f"\nEvaluating {name} ...")

    # Predictions on train and test
    y_train_pred = fitted_pipe.predict(X_train)
    y_test_pred  = fitted_pipe.predict(X_test)

    train_metrics = get_regression_metrics(y_train, y_train_pred, model_name=f"{name} (Train)", verbose=True, plot=False)
    test_metrics  = get_regression_metrics(y_test,  y_test_pred,  model_name=f"{name} (Test)",  verbose=True, plot=True)

    results[name] = {'best_params': best_params[name], 'train': train_metrics, 'test': test_metrics}


In [ ]:
for col in X_train.select_dtypes(include=np.number).columns:
    if np.allclose(X_train[col].values, y_train.values):
        print("LEAKAGE COLUMN:", col)


In [ ]:
corr = X_train.select_dtypes(include=np.number).corrwith(y_train)
print(corr.sort_values(ascending=False).head(10))


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.utils import shuffle

y_shuffled = shuffle(y_train)
scores = cross_val_score(fitted_best_estimators["Random Forest"], X_train, y_shuffled, cv=3, scoring='r2')
print(scores.mean())


In [ ]:
from sklearn.dummy import DummyRegressor
dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)
print(r2_score(y_test, dummy.predict(X_test)))


🚨 HARD LEAKAGE (remove):

booking_down_pymnt, emi_starts_from, broker_quote   (if derived from price)


Borderline (keep with caution)

yr_mfr, kms_run, total_owners, times_viewed

| Feature Type | Should You Use? | Why                  |
| ------------ | --------------- | -------------------- |
| Down payment | ❌               | Derived from price   |
| EMI          | ❌               | Algebraically linked |
| Broker quote | ❌               | Human price estimate |
| Year         | ✅               | True causal          |
| Mileage      | ✅               | True causal          |
| Owners       | ✅               | True causal          |
| Views        | ⚠️              | Proxy demand         |


In [ ]:
# identify columns with nearly unique values
n_rows = X_train.shape[0]
unique_counts = X_train.nunique(dropna=False).sort_values(ascending=False)
print(unique_counts.head(30))

# Columns with unique count equal to number of rows
unique_cols = unique_counts[unique_counts >= 0.99 * n_rows].index.tolist()
print("High-cardinality / unique-like cols:", unique_cols)


In [ ]:
from sklearn.linear_model import LinearRegression
suspect_features = []
for col in X_train.select_dtypes(include=np.number).columns:
    xi = X_train[[col]].fillna(0).values
    if np.unique(xi).size < 2:
        continue
    r2 = LinearRegression().fit(xi, y_train).score(xi, y_train)
    if r2 > 0.95:
        suspect_features.append((col, r2))
suspect_features[:20], len(suspect_features)


In [ ]:
import pandas as pd
high_corr_cat = []
for col in X_train.select_dtypes(include='object').columns:
    grp = pd.concat([X_train[col], y_train], axis=1).groupby(col)[y_train.name].mean()
    mapped = X_train[col].map(grp).fillna(y_train.mean()).values.reshape(-1,1)
    r2 = LinearRegression().fit(mapped, y_train).score(mapped, y_train)
    if r2 > 0.95:
        high_corr_cat.append((col, r2))
high_corr_cat


In [ ]:
from sklearn.metrics import r2_score
lin_relations = []
for col in X_train.columns:
    try:
        xi = X_train[[col]].fillna(0).values.astype(float)
    except Exception:
        continue
    if np.unique(xi).size < 2:
        continue
    a = np.linalg.lstsq(np.hstack([xi, np.ones_like(xi)]), y_train.values, rcond=None)[0]
    y_pred_lin = xi * a[0] + a[1]
    r2 = r2_score(y_train, y_pred_lin)
    if r2 > 0.95:
        lin_relations.append((col, float(r2), float(a[0]), float(a[1])))
lin_relations


In [ ]:
print(y_train.describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99]))
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.boxplot(y_train)
plt.title("y_train boxplot")
plt.show()


In [ ]:
from scipy.stats import ks_2samp
shifted = []
for col in X_train.select_dtypes(include=np.number).columns:
    stat, p = ks_2samp(X_train[col].dropna(), X_test[col].dropna())
    if p < 0.01:    # significant difference
        shifted.append((col, float(stat), float(p)))
len(shifted), shifted[:20]


In [ ]:
df = df.sort_values("ad_created_on")

train = df.iloc[:int(0.8*len(df))]
test  = df.iloc[int(0.8*len(df)):]


In [ ]:
df['ad_created_on']

In [ ]:
df["ad_created_on"] = pd.to_datetime(df["ad_created_on"], errors="coerce")

In [ ]:
df["ad_created_on"]

In [ ]:
df["year"] = df["ad_created_on"].dt.year
df["month"] = df["ad_created_on"].dt.month
df["dayofweek"] = df["ad_created_on"].dt.dayofweek
df["hour"] = df["ad_created_on"].dt.hour

In [ ]:
df = df.drop(columns=["ad_created_on"])

In [ ]:
print(df[["year","month"]].corrwith(df["sale_price"]).sort_values())

In [ ]:
X = df.drop(columns = 'sale_price')
y = df['sale_price']

In [ ]:
leak_features = ["booking_down_pymnt", "emi_starts_from", "broker_quote"]
X = X.drop(columns=leak_features)

In [ ]:
df_obj = X.select_dtypes(include='object')
df_obj_one = []
for col in df_obj.columns:
    unique_vals = df[col].dropna().unique()
    n_unique = len(unique_vals)

    if n_unique < 10:
        print(f"\nColumn: {col}")
        df_obj_one.append(col)
        print(f"Number of unique values: {n_unique}")
        print("Unique values:")
        print(unique_vals)
print(df_obj_one)
df_obj_target = [x for x in df_obj.columns if x not in df_obj_one]
df_obj_target
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 42,
    shuffle=False
)
# now we make the pipelines again:

num_features = X.select_dtypes(include='number')
num_features = num_features.columns
num_features # numeric

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipeline_one = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one', OneHotEncoder(handle_unknown='ignore',drop='first'))
])
cat_pipeline_target = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('target', TargetEncoder(handle_unknown='ignore',smoothing=5)),
    ('final_imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, num_features),
    ('cat_one',cat_pipeline_one,df_obj_one),
    ('cat_target',cat_pipeline_target,df_obj_target)
])
models = {
    "Random Forest": RandomForestRegressor(),
    "Linear Regression": LinearRegression(),
    "Ridge":Ridge(alpha=1.0),
    "Lasso":Lasso(),
    "ElasticNet":ElasticNet(),
    "SVR": SVR(),
    "Decision Tree": DecisionTreeRegressor()
}
pipelines = {name: Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
]) for name, model in models.items()}

In [ ]:

randomcv_models = [
    ("Decision Tree", DecisionTreeRegressor(random_state=42), dt_params),
    ("Random Forest", RandomForestRegressor(random_state=42), rf_params),
    ("Ridge", Ridge(), ridge_params),
    ("Lasso", Lasso(), lasso_params),
    ("ElasticNet", ElasticNet(), elastic_params)
]

# store fitted best estimators and best params
fitted_best_estimators = {}
best_params = {}

for name, model, params in randomcv_models:
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    params_prefixed = {f"model__{k}": v for k, v in params.items()}

    random_search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=params_prefixed,
        n_iter=50,
        cv=3,
        verbose=2,
        n_jobs=-1,
        random_state=42,
        scoring='r2',
        refit=True  # ensure best_estimator_ is fitted on the whole X_train
    )

    random_search.fit(X_train, y_train)       # fitted on X_train folds
    print(f"BEST for {name}: {random_search.best_params_}")
    best_params[name] = random_search.best_params_
    # best_estimator_ is already a Pipeline fitted on the entire X_train
    fitted_best_estimators[name] = random_search.best_estimator_

# Evaluate each fitted best estimator
results = {}
for name, fitted_pipe in fitted_best_estimators.items():
    print(f"\nEvaluating {name} ...")

    # Predictions on train and test
    y_train_pred = fitted_pipe.predict(X_train)
    y_test_pred  = fitted_pipe.predict(X_test)

    train_metrics = get_regression_metrics(y_train, y_train_pred, model_name=f"{name} (Train)", verbose=True, plot=False)
    test_metrics  = get_regression_metrics(y_test,  y_test_pred,  model_name=f"{name} (Test)",  verbose=True, plot=True)

    results[name] = {'best_params': best_params[name], 'train': train_metrics, 'test': test_metrics}


Short interpretation of the key metrics (per model)

Numbers you posted (selecting the test RMSE/R² and train vs test comparison):

Decision Tree

Train R² = 0.9665, Test R² = 0.7296

RMSE (test) ≈ 144,232

Interpretation: tree is overfitting (very high train R²) but still gives decent test performance. High variance model.

Random Forest

Train R² = 0.9849, Test R² = 0.8082

RMSE (test) ≈ 121,460

Interpretation: ensemble reduced variance vs a single tree and generalizes best among your models. Still some overfit (train R² very high) but test performance is the best.

Ridge / Lasso / ElasticNet (linear models)

Train R² ≈ 0.71, Test R² ≈ 0.68

RMSE (test) ≈ 156k (similar across these)

Interpretation: linear models explain less variance. Regularization (L1/L2) yields similar performance — suggests relationships aren’t purely linear or linear models can’t capture complex dependencies.

Put errors into context (useful perspective)

Median sale price ≈ 384,849 (from your y_train.describe()).

RF test RMSE 121k → relative RMSE ≈ 31.5% of median price.

RF MAE 54k → mean absolute error ≈ 14% of median price.

These are realistic ballpark errors for car resale price prediction (depending on problem difficulty). RF doing ~0.81 R² is good.

In [ ]:
# better alternative:
df['ad_created_on'] = pd.to_datetime(df['ad_created_on'])
max_date = df['ad_created_on'].max()
df['listing_age_days'] = (max_date - df['ad_created_on']).dt.days
# drop ad_created_on afterwards

# and cyclic features:
# month in 1..12
df['month_sin'] = np.sin(2*np.pi * (df['month']-1) / 12)
df['month_cos'] = np.cos(2*np.pi * (df['month']-1) / 12)

# hour in 0..23
df['hour_sin'] = np.sin(2*np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2*np.pi * df['hour'] / 24)

# dayofweek in 0..6
df['dow_sin'] = np.sin(2*np.pi * df['dayofweek'] / 7)
df['dow_cos'] = np.cos(2*np.pi * df['dayofweek'] / 7)



### interpretation:
1) Short interpretation of the key metrics (per model)

Numbers you posted (selecting the test RMSE/R² and train vs test comparison):

Decision Tree

Train R² = 0.9665, Test R² = 0.7296

RMSE (test) ≈ 144,232

Interpretation: tree is overfitting (very high train R²) but still gives decent test performance. High variance model.

Random Forest

Train R² = 0.9849, Test R² = 0.8082

RMSE (test) ≈ 121,460

Interpretation: ensemble reduced variance vs a single tree and generalizes best among your models. Still some overfit (train R² very high) but test performance is the best.

Ridge / Lasso / ElasticNet (linear models)

Train R² ≈ 0.71, Test R² ≈ 0.68

RMSE (test) ≈ 156k (similar across these)

Interpretation: linear models explain less variance. Regularization (L1/L2) yields similar performance — suggests relationships aren’t purely linear or linear models can’t capture complex dependencies.

Put errors into context (useful perspective)

Median sale price ≈ 384,849 (from your y_train.describe()).

RF test RMSE 121k → relative RMSE ≈ 31.5% of median price.

RF MAE 54k → mean absolute error ≈ 14% of median price.

These are realistic ballpark errors for car resale price prediction (depending on problem difficulty). RF doing ~0.81 R² is good.

2) Why these behaviours occur (short, technical)

Decision Tree memorizes idiosyncrasies (overfits); pruning parameters (min_samples_leaf / max_depth) were tuned but still allow high train fit.

Random Forest reduces variance by averaging many trees — that’s why test R² improved. The chosen max_features=5 means each split uses a small random subset of features (strong regularization), bootstrap=False means each tree trained on entire sample (rare choice; it reduces randomness but in your case still produced good generalization).

Linear models do not capture non-linear interactions and high-cardinality categorical structure (unless well encoded). They are stable but lower performing here.

Regularization picks (alpha): large alphas for Lasso indicate strong shrinkage; ElasticNet tuned to near-L1 (l1_ratio ~0.9) meaning sparse-ish solution, but these still underperform compared to RF.

3) Will the new columns (year, month, dayofweek, hour) be processed by your pipeline?

Yes — in your current code they will be processed — and as numeric features.

Why:

You created year/month/dayofweek/hour on df before creating X = df.drop('sale_price').

Then you computed:

num_features = X.select_dtypes(include='number').columns


So these new columns (integers) are included in num_features.

In your ColumnTransformer you assigned ('numeric', numeric_pipeline, num_features). That numeric pipeline (SimpleImputer then StandardScaler) will be applied to them during fit/transform.

So in short: they are treated as numeric features and will be imputed & scaled.

4) Recommendations for time-derived features (what to change and why)

Numeric treatment (your current) is acceptable, but you can do better:

A — Use age or relative time rather than raw year

Raw year often captures market trend; better: compute age relative to the most recent listing in the dataset:

df['ad_created_on'] = pd.to_datetime(df['ad_created_on'])
max_date = df['ad_created_on'].max()
df['listing_age_days'] = (max_date - df['ad_created_on']).dt.days
# drop ad_created_on afterwards


This reduces raw trend leakage and makes the feature interpretable.

B — Encode cyclical features for month, hour, dayofweek

Months/hours/day-of-week are cyclical; representing them as integers introduces artificial discontinuities (e.g., 12→1). Convert to sin/cos pairs:

# month in 1..12
df['month_sin'] = np.sin(2*np.pi * (df['month']-1) / 12)
df['month_cos'] = np.cos(2*np.pi * (df['month']-1) / 12)

# hour in 0..23
df['hour_sin'] = np.sin(2*np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2*np.pi * df['hour'] / 24)

# dayofweek in 0..6
df['dow_sin'] = np.sin(2*np.pi * df['dayofweek'] / 7)
df['dow_cos'] = np.cos(2*np.pi * df['dayofweek'] / 7)


Keep or drop the integer columns after creating sin/cos. The sin/cos remain numeric and will be scaled in your numeric pipeline.

C — Decide if year should be numeric or categorical

If year is small-range (e.g., 2015–2022), numeric is fine.

If year has few categories and you want non-linear interactions, you could treat it as categorical (one-hot or target-encode), but numeric/age is usually better.

5) If you want these time features to be treated as categorical in your pipeline

Convert integer columns to object (strings) before select_dtypes and include them in df_obj_one or df_obj_target. Example:

X['month'] = X['month'].astype(str)
# then your df_obj detection will pick it up


But for cyclical time info: do not one-hot month (that creates more columns and loses continuity). Use sin/cos.

6) Important caution about inner CV vs time split (one big gotcha)

You used train_test_split(..., shuffle=False) and sorted df by ad_created_on — good for producing a realistic time-based holdout.

BUT your RandomizedSearchCV(..., cv=3) by default uses KFold (not time-aware). That will split X_train into folds that may mix future and past inside inner CV, creating temporal leakage inside hyperparameter tuning. For a time series / time-ordered dataset you should use a time-aware CV like:

from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=3)
random_search = RandomizedSearchCV(..., cv=tscv, ...)


Use TimeSeriesSplit (or custom expanding-window splits) to make inner CV respect chronology. This is essential when using target-encoding too.

7) Suggested actionable changes (code snippets)

A. Create age + cyclical features and drop raw timestamp:

df['ad_created_on'] = pd.to_datetime(df['ad_created_on'])
max_date = df['ad_created_on'].max()
df['listing_age_days'] = (max_date - df['ad_created_on']).dt.days
df['month'] = df['ad_created_on'].dt.month
df['hour'] = df['ad_created_on'].dt.hour
df['dayofweek'] = df['ad_created_on'].dt.dayofweek

# cyclical
df['month_sin'] = np.sin(2*np.pi*(df['month']-1)/12)
df['month_cos'] = np.cos(2*np.pi*(df['month']-1)/12)
# similar for hour/dayofweek...

df = df.drop(columns=['ad_created_on','month','hour','dayofweek'])  # if you want only sin/cos and age


B. Use TimeSeriesSplit inside RandomizedSearchCV:

from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=3)
RandomizedSearchCV(..., cv=tscv, ...)


C. Consider log-transforming the target to reduce skew:

y_train_log = np.log1p(y_train)
# fit models on y_train_log, predict y_pred_log
y_pred = np.expm1(y_pred_log)  # inverse before computing MAE/RMSE on original units


D. If OneHotEncoder explodes features, replace some OHE usage with HashingEncoder or keep target encoders that are CV-safe (but use TimeSeriesSplit for inner CV to be safe).

8) Ways to further improve model performance

Try CatBoostRegressor or LightGBM (both handle categorical features better and often beat RF on tabular data).

Use feature selection or remove extremely sparse high-cardinality columns.

Add interaction features like age * kms_run or age^2 if domain knowledge supports it.

Train on log1p(y) to stabilize errors and then invert predictions for realistic RMSE/MAE computation.

Produce a learning curve (train vs validation error vs training size) to check if adding more data helps or if model saturates.

9) Small notes about your hyperparameters shown

RandomForest bootstrap=False: unusual (disables bagging). Often bootstrap=True is used — try both.

max_features=5 chosen by search: good regularization when you have many features.

Lasso regularization large alpha=100: means heavy feature shrinkage; if Lasso drove coefficients to zero that explains its similar performance to Ridge in your run.

#### Actions:
Replace raw year/month/dayofweek/hour with listing_age_days and cyclical sin/cos features. Drop raw timestamp.

Use TimeSeriesSplit for RandomizedSearchCV.

Retrain Random Forest / CatBoost on log1p(y) and report inverted metrics with np.expm1.

Run permutation importance or SHAP on RF/CB to verify feature importance — make sure time-related features aren't dominating (if they are, consider different splits or remove trend features).

## Time series aware:

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=3)
randomcv_models = [
    ("Decision Tree", DecisionTreeRegressor(random_state=42), dt_params),
    ("Random Forest", RandomForestRegressor(random_state=42), rf_params),
    ("Ridge", Ridge(), ridge_params),
    ("Lasso", Lasso(), lasso_params),
    ("ElasticNet", ElasticNet(), elastic_params)
]

# store fitted best estimators and best params
fitted_best_estimators = {}
best_params = {}

for name, model, params in randomcv_models:
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    params_prefixed = {f"model__{k}": v for k, v in params.items()}

    random_search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=params_prefixed,
        n_iter=50,
        cv=tscv,
        verbose=2,
        n_jobs=-1,
        random_state=42,
        scoring='r2',
        refit=True  # ensure best_estimator_ is fitted on the whole X_train
    )

    random_search.fit(X_train, y_train)       # fitted on X_train folds
    print(f"BEST for {name}: {random_search.best_params_}")
    best_params[name] = random_search.best_params_
    # best_estimator_ is already a Pipeline fitted on the entire X_train
    fitted_best_estimators[name] = random_search.best_estimator_

# Evaluate each fitted best estimator
results = {}
for name, fitted_pipe in fitted_best_estimators.items():
    print(f"\nEvaluating {name} ...")

    # Predictions on train and test
    y_train_pred = fitted_pipe.predict(X_train)
    y_test_pred  = fitted_pipe.predict(X_test)

    train_metrics = get_regression_metrics(y_train, y_train_pred, model_name=f"{name} (Train)", verbose=True, plot=False)
    test_metrics  = get_regression_metrics(y_test,  y_test_pred,  model_name=f"{name} (Test)",  verbose=True, plot=True)

    results[name] = {'best_params': best_params[name], 'train': train_metrics, 'test': test_metrics}


# Learning and IMP takeaways:
You’re likely referring to **time-based encoding** (feature engineering for time series). I’ll explain **what it is, why it helps ML models, and practical rules you must follow**—especially since this was your first exposure to time-series data.

---

# 1) What is Time Encoding (in Time Series)?

Raw time columns (timestamps) are **not directly usable by ML models**.

Example raw data:

```
2024-01-01 10:23:45
2024-01-01 10:24:45
```

Models **do not understand time semantics**—they just see arbitrary strings or integers.

So we **encode time into meaningful numerical features**.

---

# 2) Common Time-Based Encodings

## ✅ A) Calendar Decomposition (Basic & Mandatory)

From timestamp extract:

* Year
* Month
* Day
* Hour
* Minute
* Second
* Day of week
* Is weekend
* Week of year

### Example:

```
2024-01-01 10:23:45
→ Year=2024
→ Month=1
→ Day=1
→ Hour=10
→ DayOfWeek=0 (Monday)
```

### Why?

Because **patterns differ by hour, day, season**.

---

## ✅ B) Cyclical Encoding (Very Important)

Time is **circular**.

Example:

* Hour 23 and Hour 0 are close
* December and January are close

If you use normal integers:

```
Hour 23 vs Hour 0 → model thinks they are far apart
```

### Solution: Sine & Cosine Encoding

For hour:

[
hour_sin = sin(2π * hour / 24)
]
[
hour_cos = cos(2π * hour / 24)
]

Same for:

* Day of week (7)
* Month (12)
* Minute (60)

### Why?

Preserves **circular continuity**.

---

## ✅ C) Lag Features (Core Time Series Concept)

Time series depends on **past values**.

Create:

```
lag_1 = value at t-1
lag_2 = value at t-2
lag_24 = value 24 steps ago
```

### Example:

Electricity demand today depends on yesterday and last week.

---

## ✅ D) Rolling Window Statistics

Capture trends:

```
rolling_mean_7
rolling_std_30
rolling_max_24
```

These capture:

* Trend
* Seasonality
* Volatility

---

# 3) Why Encoding Helps ML Models

ML models assume **IID data (independent samples)**.
Time series is **NOT IID**.

Encoding helps models learn:

| Pattern Type    | Encoding Used |
| --------------- | ------------- |
| Daily cycle     | Hour sin/cos  |
| Weekly cycle    | Day-of-week   |
| Seasonal        | Month         |
| Trend           | Rolling mean  |
| Autocorrelation | Lag features  |

Without encoding → model becomes blind to temporal structure.

---

# 4) Critical Time Series Rules (YOU MUST REMEMBER)

These are **non-negotiable**.

---

## ⚠️ Rule 1: NEVER Shuffle Time Series Data

Wrong:

```python
train_test_split(shuffle=True)
```

Correct:

```python
train = data[:80%]
test = data[80%:]
```

Why?
Future data must not leak into past.

---

## ⚠️ Rule 2: No Data Leakage in Rolling/Lag Features

Rolling windows must use **past only**, never future.

---

## ⚠️ Rule 3: Scaling Must Be Fit Only on Train Data

Wrong:

```python
scaler.fit(full_data)
```

Correct:

```python
scaler.fit(train)
```

---

## ⚠️ Rule 4: Cross Validation Must Be Time-Aware

Use:

* TimeSeriesSplit
* Walk-forward validation

NOT k-fold.

---

# 5) How Time Series Differs from Normal ML Data

| Normal Dataset   | Time Series Dataset    |
| ---------------- | ---------------------- |
| Rows independent | Rows dependent on past |
| Shuffle allowed  | Shuffle forbidden      |
| Random CV        | Time-aware CV          |
| Static features  | Temporal features      |

---

# 6) When You MUST Use Time Encoding

Any dataset with:

* Timestamp
* Sequential order
* Stock prices
* Sensor readings
* Website traffic
* Weather
* Logs
* User activity sequences

---

# 7) Minimal Practical Pipeline (What You Should Do)

1. Convert timestamp to datetime
2. Extract calendar features
3. Add cyclical sin/cos
4. Create lag features
5. Create rolling stats
6. Split by time
7. Scale using train only
8. Train model

---

# 8) Why Your Model Improved After Encoding

Because you **converted time into structured signals**.

Instead of:

```
2024-01-01 10:00
2024-01-01 11:00
```

Model now sees:

```
hour_sin, hour_cos, lag_1, rolling_mean
```

Which **directly encode patterns**.

---

# 9) Advanced Tip (Industry Level)

Deep models (LSTM, Transformers) learn time dependencies automatically,
BUT feature engineering still improves:

* Training stability
* Data efficiency
* Interpretability

---

# If You Want

I can give you:

✅ Code template for full time-series feature engineering
✅ Visual intuition of cyclical encoding
✅ Interview-ready explanation for “How to handle time-series data”
✅ Common beginner mistakes (and how you likely made them)
✅ Mathematical intuition of autocorrelation and stationarity

Just tell me your dataset type (stock, sensor, tabular logs, etc.).


1) Why Sine & Cosine Are Used for Time (Core Intuition)

Many time variables are cyclical:

Hour: 0 → 23 → back to 0

Day of week: Mon → Sun → Mon

Month: Jan → Dec → Jan

If you encode them as integers, you break the geometry of time.

❌ Problem with Normal Integer Encoding

Example: Hour of day

Hour 23
Hour 0


Numeric encoding:

23 and 0 → distance = 23


But in reality:

23:00 and 00:00 are only 1 hour apart


So the model learns wrong relationships.

2) Mathematical Idea Behind Sin/Cos Encoding

A cycle is best represented as a circle.

To map a cycle to a circle:

𝑥
𝑠
𝑖
𝑛
=
sin
⁡
(
2
𝜋
⋅
𝑡
𝑇
)
x
sin
	​

=sin(2π⋅
T
t
	​

)
𝑥
𝑐
𝑜
𝑠
=
cos
⁡
(
2
𝜋
⋅
𝑡
𝑇
)
x
cos
	​

=cos(2π⋅
T
t
	​

)

Where:

𝑡
t = time value (hour, day, month)

𝑇
T = period (24 for hours, 7 for days, 12 for months)

Example: Hour Encoding

For hour = 0:

𝑠
𝑖
𝑛
(
0
)
=
0
,
𝑐
𝑜
𝑠
(
0
)
=
1
sin(0)=0,cos(0)=1

For hour = 23:

𝑠
𝑖
𝑛
(
2
𝜋
∗
23
/
24
)
≈
−
0.2588
sin(2π∗23/24)≈−0.2588
𝑐
𝑜
𝑠
(
2
𝜋
∗
23
/
24
)
≈
0.9659
cos(2π∗23/24)≈0.9659

For hour = 1:

𝑠
𝑖
𝑛
(
2
𝜋
∗
1
/
24
)
≈
0.2588
sin(2π∗1/24)≈0.2588
𝑐
𝑜
𝑠
(
2
𝜋
∗
1
/
24
)
≈
0.9659
cos(2π∗1/24)≈0.9659

👉 Notice hour 23 and hour 1 are close in cosine value → model understands proximity.

3) Why Both Sin AND Cos?

Because sine alone is ambiguous.

Example:

sin(0°) = 0
sin(180°) = 0   (different time but same sin)


Cosine resolves ambiguity.

Together they form unique circular coordinates.

4) Geometric Interpretation (Important)

Each time value becomes a point on a unit circle.

(hour_sin, hour_cos)


So:

23:00 and 00:00 are neighbors in 2D space

Noon and midnight are opposite points

This preserves cyclic continuity.

5) Why ML Models Care About This

ML models assume Euclidean distance.

Integer encoding → linear distance
Sin/cos encoding → circular distance

So models learn:

Daily seasonality

Weekly seasonality

## EDA for another dataset do use these graphs and methods

In [ ]:
# Numerical features:
num_features = df.select_dtypes(include='number').columns.tolist()
print(f'Number of Numerical Features: {len(num_features)}')

# categorical features:
cat_features = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Number of Categorical Features: {len(cat_features)}')

# Discrete features:
discrete_features = [
    f for f in num_features if df[f].nunique() <= 25
]
print(f'Number of Discrete Features: {len(discrete_features)}')

# Continuous features:
con_features = [
    f for f in num_features if f not in discrete_features
]
print(f'Number of Continuous Features: {len(con_features)}')

In [ ]:
# Getting all Different Types OF Features
num_features = [feature for feature in df.columns if df[feature].dtype != 'O' ]
print('Num of Numerical Features :', len(num_features))
cat_features = [feature for feature in df.columns if df[feature].dtype == 'O' ]
print('Num features of Categorical Features :', len(cat_features))
discrete_features=[feature for feature in num_features if len(df[feature].unique()) <= 25]
print('Num of Discrete Features :',len(discrete_features))
continuous_features=[feature for feature in num_features if feature not in discrete_features]
print('Num of Continuous Features :',len(continuous_features))

In [ ]:
df["name_2"] = df.name.apply(lambda x : ' '.join(x.split(' ')[:1]))
df['name_2']

In [ ]:
len(df.name_2.value_counts())

In [ ]:
sns.countplot(data=df,x="name_2",palette="CMRmap")
plt.xticks(rotation=90)
plt.xlabel("Name",fontsize=10,color="black")
plt.ylabel("Name",fontsize=10,color="black")
plt.title("NAME COUNT",color="black")
plt.show()

In [ ]:
labels = df["name_2"][:30].value_counts().index
sizes = df["name_2"][:30].value_counts()
colors = ['#ff9999','#66b3ff','#99ff99','#ffcc99',"pink","yellow"]
plt.figure(figsize = (8,8))
plt.pie(sizes, labels=labels, rotatelabels=False, autopct='%1.1f%%',colors=colors,shadow=True, startangle=45)
plt.title('name',color = 'red',fontsize = 15)
plt.show()

In [ ]:
%pip install -r C:\xtra\Last_Chance\git_re\inexorable-ML\requirements.txt

In [ ]:
from wordcloud import WordCloud, STOPWORDS

text = ' '.join(df['name_2'])

plt.rcParams['figure.figsize'] = (12,12)
wordcloud = WordCloud(background_color = 'black',colormap='vlag', width = 1200,  height = 1200, max_words = 121).generate(text)
plt.imshow(wordcloud)
plt.axis('off')
plt.show()
plt.show()

In [ ]:
sns.countplot(data=df,x="year",palette="icefire")
plt.xticks(rotation=90)
plt.xlabel("YEAR",fontsize=10,color="RED")
plt.ylabel("COUNT",fontsize=10,color="RED")
plt.title("YEAR COUNT",color="RED")
plt.show()

In [ ]:
labels = df["year"][:40].value_counts().index
sizes = df["year"][:40].value_counts()
colors = ['#ff9999','#66b3ff','#99ff99','#ffcc99',"pink","yellow"]
plt.figure(figsize = (8,8))
plt.pie(sizes, labels=labels, rotatelabels=False, autopct='%1.1f%%',colors=colors,shadow=True, startangle=45)
plt.title('Year',color = 'red',fontsize = 15)
plt.show()

In [ ]:
df.fuel.value_counts()

In [ ]:
labels = df["fuel"].value_counts().index
sizes = df["fuel"].value_counts()
colors = ['#ff9999','#66b3ff','#99ff99','#ffcc99',"pink","yellow"]
plt.figure(figsize = (8,8))
plt.pie(sizes, labels=labels, rotatelabels=False, autopct='%1.f%%',colors=colors,shadow=True, startangle=9)
plt.title('fuel',color = 'blue',fontsize = 15)
plt.show()

In [ ]:
sns.countplot(data=df,x="seller_type",palette="pink")
plt.xlabel("SELLER TYPE",fontsize=10,color="brown")
plt.ylabel("COUNT",fontsize=10,color="brown")
plt.title("SELLER TYPE COUNT",color="brown")
plt.show()

In [ ]:
sns.countplot(data=df,x="transmission",palette="Spectral")
plt.xlabel("TRANMISSION",fontsize=10,color="GREEN")
plt.ylabel("COUNT",fontsize=10,color="GREEN")
plt.title("TRANMISSION COUNT",color="GREEN")
plt.show()

In [ ]:
sns.histplot(data=df, x="year", hue="transmission")
plt.xticks(rotation=45)
plt.show()

In [ ]:
df1 = df.groupby(["transmission","fuel","name_2"],as_index=False)[['selling_price']].median().rename(columns={'selling_price':'price'})
fig = px.treemap(df1, path = [px.Constant("all"), "transmission","fuel","name_2"],
                 values   ='price', color='name_2',
                 color_discrete_map={'(?)':'lightgrey', 'Lunch':'gold', 'Dinner':'darkblue'})
fig.update_layout(margin  = dict(t=50, l=25, r=25, b=25))
fig.show()

In [ ]:
pd.crosstab(df["name_2"], df["transmission"]).plot(kind="bar", figsize=(10, 6), color=["blue","red"], title="name and transmission ")
plt.show()

In [ ]:
plt.style.use("fivethirtyeight")
plt.figure(figsize=(10,10))
plt.title("happiness rate of continents by year")
sns.set(font_scale=1)
plt.xticks(rotation=90)
sns.barplot(data=df, x="name_2", y="km_driven",hue="transmission",palette="gist_rainbow")
plt.show()

In [ ]:
joint_data=df.sort_values(by='year', ascending=False)

top_rated=joint_data[:2500]
fig =px.sunburst(
    top_rated,
    path=['year',"name_2"],
    values='year',
    color='year')
fig.show()

In [ ]:
joint_data=df.sort_values(by='year', ascending=False)

top_rated=joint_data[2500:]
fig =px.sunburst(
    top_rated,
    path=['year',"name_2"],
    values='year',
    color='year')
fig.show()

In [ ]:
fig = px.strip(df, x='year', y='name_2', color='year')
fig.show()

In [ ]:
f, ax = plt.subplots(figsize=(20,8))
sns.boxplot(x=df["name_2"].values, y = df["km_driven"].values,palette="twilight",ax=ax)
plt.xticks(rotation=90)
plt.show()

In [ ]:
#Indpendent and dependent features
from sklearn.model_selection import train_test_split
X = df.drop(['selling_price'], axis=1)
y = df['selling_price']

In [ ]:
df['name'].value_counts()

In [ ]:
X.head()